In [ ]:
import json
import os
import re
import pandas as pd

from phonology_convertion import is_Vietnamese

In [22]:
def load_json(path):
    return json.load(open(path, 'r', encoding='utf-8'))


def save_json(data, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

In [23]:
daiphatthanh_path = r'C:\Users\ASUS\Desktop\crawl_fb\src\data\comment\daiphatthanh_comments.json'
day_path = r'C:\Users\ASUS\Desktop\crawl_fb\src\data\comment\day1000_comments.json'
theanh_path = r'C:\Users\ASUS\Desktop\crawl_fb\src\data\comment\theanh_comments.json'

daiphatthanh = load_json(daiphatthanh_path)['comments']
day = load_json(day_path)['comments']
theanh = load_json(theanh_path)['comments']

len(daiphatthanh), len(day), len(theanh)

(5297, 1381, 1914)

In [ ]:
# import re


# def remove_user_mentions(text):
#     """
#     Xoá các username kiểu:
#     @abc
#     @abc_123
#     @nhuy_25_
#     @huonggiang
#     @miss.count
#     """
#     return re.sub(r"@\w[\w.]*", " ", text, flags=re.UNICODE)


# def remove_emoji(text):
#     emoji_pattern = re.compile(
#         "["
#         "\U0001F1E6-\U0001F1FF"  # flags
#         "\U0001F300-\U0001F5FF"  # symbols & pictographs
#         "\U0001F600-\U0001F64F"  # emoticons
#         "\U0001F680-\U0001F6FF"  # transport & map
#         "\U0001F700-\U0001F77F"
#         "\U0001F780-\U0001F7FF"
#         "\U0001F800-\U0001F8FF"
#         "\U0001F900-\U0001F9FF"
#         "\U0001FA00-\U0001FA6F"
#         "\U0001FA70-\U0001FAFF"
#         "\U00002700-\U000027BF"  # dingbats
#         "\U00002600-\U000026FF"  # misc symbols
#         "\U00002300-\U000023FF"  # technical symbols
#         "\U0000203C-\U00002049"  # ‼️ ⁉️
#         "\U00002B00-\U00002BFF"  # arrows/stars
#         "\U00003030"
#         "\U0000303D"
#         "\U0000FE0F"             # variation selector
#         "\U0000200D"             # zero width joiner
#         "]+",
#         flags=re.UNICODE
#     )

#     return emoji_pattern.sub(" ", text)


# def normalize_emoticon(text):
#     """
#     Chuẩn hoá text emoticon:
#     :))))))) -> :))
#     :((((((( -> :((
#     """
#     text = re.sub(r"(:-?\){2})\)+", r"\1", text)
#     text = re.sub(r"(:-?\({2})\(+", r"\1", text)
#     return text


# def split_sentences(text):
#     sentences = re.split(r'(?<=[.!?])\s+', text.strip())
#     return [s.strip() for s in sentences if s.strip()]


# def extract_words(sentence):
#     return re.findall(r"[A-Za-zÀ-ỹĐđ]+", sentence)


# def is_vietnamese_word(word):
#     word = word.lower().strip()

#     if not word:
#         return False

#     try:
#         res, _ = is_Vietnamese(word)
#         return res

#     except Exception:
#         return False
    
# def remove_urls(text):
#     """
#     Xoá các đường link kiểu:
#     https://abc.com
#     http://abc.com
#     www.abc.com
#     abc.com/path
#     tiktok.com/@user/video/...
#     """
#     url_pattern = r"""
#         (https?://\S+)              # http:// hoặc https://
#         |(www\.\S+)                 # www.
#         |([a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\S*)  # domain.com/path
#     """
#     return re.sub(url_pattern, " ", text, flags=re.VERBOSE)


# def preprocess_data(data, min_words=4):
#     res = []

#     for cmt in data:
#         if cmt is None:
#             continue

#         cmt = str(cmt)

#         # 1. Xoá link
#         cmt = remove_urls(cmt)

#         # 2. Xoá username kiểu @abc, @abc_123
#         cmt = remove_user_mentions(cmt)

#         # 3. Chuẩn hoá text emoticon kiểu :))))
#         cmt = normalize_emoticon(cmt)

#         # 4. Bỏ emoji thật, kể cả ‼️ ❗ ❤️
#         cmt = remove_emoji(cmt)

#         # 5. Chuẩn hóa khoảng trắng
#         cmt = re.sub(r"\s+", " ", cmt).strip()

#         if not cmt:
#             continue

#         # 6. Sentence split
#         sentences = split_sentences(cmt)

#         for sent in sentences:
#             words = extract_words(sent)

#             if len(words) < min_words:
#                 continue

#             has_non_vietnamese_word = any(
#                 not is_vietnamese_word(word)
#                 for word in words
#             )

#             if not has_non_vietnamese_word:
#                 continue

#             res.append(sent)

#     return res

In [35]:
import re
import unicodedata


def normalize_unicode_text(text):
    """
    Chuyển fancy unicode về dạng thường:
    𝐖𝐎𝐖 -> WOW
    𝐆Á𝐈 -> GÁI
    """
    text = unicodedata.normalize("NFKC", text)
    text = unicodedata.normalize("NFC", text)
    return text


def is_latin_char(ch):
    """
    Giữ chữ Latin, bao gồm tiếng Việt có dấu.
    """
    try:
        return "LATIN" in unicodedata.name(ch)
    except ValueError:
        return False


def remove_non_latin_noise(text):
    """
    Xoá ký tự trang trí, ký tự ngoại lai, ký tự unicode rác.
    Giữ lại:
    - chữ Latin / tiếng Việt
    - số
    - khoảng trắng
    - một số dấu câu cơ bản
    """
    allowed_punct = set(".,!?;:'\"()-")

    cleaned = []

    for ch in text:
        if ch.isspace():
            cleaned.append(" ")

        elif is_latin_char(ch):
            cleaned.append(ch)

        elif ch.isdigit():
            cleaned.append(ch)

        elif ch in allowed_punct:
            cleaned.append(ch)

        else:
            cleaned.append(" ")

    return "".join(cleaned)


def remove_user_mentions(text):
    """
    Xoá username thường:
    @abc
    @abc_123
    @nhuy_25_

    Và username trang trí:
    @꧁ CHỒNG CỦA EM ꧂:
    """
    # Xoá dạng @......:
    text = re.sub(r"@\s*[^:：\n\r]{1,100}[:：]", " ", text)

    # Xoá dạng @abc, @abc_123, @diemhang><
    text = re.sub(r"@\S+", " ", text)

    return text


def remove_urls(text):
    return re.sub(r"(https?://\S+|www\.\S+|\S+\.[a-zA-Z]{2,}\S*)", " ", text)


def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F1E6-\U0001F1FF"
        "\U0001F300-\U0001F5FF"
        "\U0001F600-\U0001F64F"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FAFF"
        "\U00002700-\U000027BF"
        "\U00002600-\U000026FF"
        "\U00002300-\U000023FF"
        "\U0000203C-\U00002049"
        "\U00002B00-\U00002BFF"
        "\U00003030"
        "\U0000303D"
        "\U0000FE0F"
        "\U0000200D"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(" ", text)


def normalize_emoticon(text):
    text = re.sub(r"(:-?\){2})\)+", r"\1", text)  # :)))) -> :))
    text = re.sub(r"(:-?\({2})\(+", r"\1", text)  # :(((( -> :((
    return text


def clean_noise_text(text):
    text = str(text)

    # 1. Xoá link
    text = remove_urls(text)

    # 2. Xoá username
    text = remove_user_mentions(text)

    # 3. Chuẩn hoá fancy unicode
    text = normalize_unicode_text(text)

    # 4. Chuẩn hoá emoticon
    # text = normalize_emoticon(text)

    # 5. Xoá emoji
    text = remove_emoji(text)

    # 6. Xoá ký tự trang trí / unicode rác
    text = remove_non_latin_noise(text)

    # 7. Chuẩn hoá khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text

def is_vietnamese_word(word):
    word = word.lower().strip()

    if not word:
        return False

    try:
        res, _ = is_Vietnamese(word)
        return res

    except Exception:
        return False

def preprocess_data(data, min_words=4):
    res = []

    for cmt in data:
        if cmt is None:
            continue

        # Clean mạnh hơn
        cmt = clean_noise_text(cmt)

        if not cmt:
            continue

        sentences = split_sentences(cmt)

        for sent in sentences:
            words = extract_words(sent)

            if len(words) < min_words:
                continue

            has_non_vietnamese_word = any(
                not is_vietnamese_word(word)
                for word in words
            )

            if not has_non_vietnamese_word:
                continue

            res.append(sent)

    return res

# Lâm

In [36]:
daiphatthanh_clean = preprocess_data(daiphatthanh)
theanh_clean = preprocess_data(theanh)
day_clean = preprocess_data(day)

In [37]:
len(daiphatthanh_clean), len(day_clean) , len(theanh_clean) , len(daiphatthanh_clean + day_clean + theanh_clean)

(2512, 438, 758, 3708)

In [90]:
len(daiphatthanh_clean), len(day_clean) , len(theanh_clean) , len(daiphatthanh_clean + day_clean + theanh_clean)

(1608, 355, 601, 2564)

In [ ]:
# save_json(daiphatthanh_clean, r'C:\Users\ASUS\Desktop\crawl_fb\src\data\clean\daiphatthanh.json')
# save_json(theanh_clean, r'C:\Users\ASUS\Desktop\crawl_fb\src\data\clean\theanh.json')
# save_json(day_clean, r'C:\Users\ASUS\Desktop\crawl_fb\src\data\clean\day.json')

# Thy Thy

In [30]:
tiktok_raw = pd.read_csv(r'C:\Users\ASUS\Desktop\crawl_fb\src\data\external\tiktok_comments_api_clean.csv')

In [31]:
titkok_raw = tiktok_raw["text"].to_list()

In [39]:
tiktok_clean = preprocess_data(titkok_raw)

In [40]:
len(tiktok_raw), len(tiktok_clean)

(18676, 8267)

In [73]:
len(tiktok_raw), len(tiktok_clean)

(18676, 6624)

In [41]:
tiktok_clean

['ảnh ko hề hơn thua',
 'tánh a ko thích hơn thua , mà cũng ko thích thua ai',
 'cái giọng cười bà nào zậy',
 'hồi còn học mẫu giáo m cũng hay vậy lắm',
 'anh k thích hơn thua mà thua thì anh k chịu',
 'Nhãn tập thiết kế game bóng đá',
 'Việt quất ngon lắm hả hay j',
 'Filter ở đâu vậy ạ',
 'cứ s s á',
 'ấn đường hơi tốt nha b',
 'Cái vị việt quất hay ăn sinh tố sirus nó khác với quả lắm mn ,nhạt chả có vị gì',
 'vd nay phải lên xh',
 'ăn gi nhanh v tr',
 'việt quất có vị như nào v mn',
 'mukbang vs 36 tư thế',
 'mukbang mọi filter hả..',
 'thấy cx hợp hợp',
 'có vậy mà t coi hết á',
 'nó quyến rũ j đâu á',
 'giòn quá shop ơi',
 'Sticker ở đây trước khi xh',
 'ê môi bả đẹp v',
 'video ưng ý nhất',
 'Filter tên j v',
 'ăn từ từ mă oi',
 'ai rượt hay ss mà ăn nhanh dậyy',
 'Nho hay j vậy?',
 'tuyệt vcl ý là bị cute mà không bị khó chịu, siêu yêu siêu mê ăn uống nhai đóng miệng không chẹp chẹp, t coi nhiều vd rồi mà t không thể ngừng nhớ tới vd này',
 'Cái này th mà t cười nãy h',
 'hộ vd

In [50]:
save_json(tiktok_clean, r'C:\Users\ASUS\Desktop\crawl_fb\src\data\clean\titok.json')

# Sơn

In [42]:
thread_raw = load_json(r'C:\Users\ASUS\Desktop\crawl_fb\src\data\external\threads_results.json')

In [43]:
thread_raw = [
    item['text'] for item in thread_raw
]

In [44]:
thread_clean = preprocess_data(thread_raw)

In [47]:
len(thread_raw), len(thread_clean)

(11219, 8915)

In [48]:
thread_clean

['ĐKM lắp theo đúng tờ hướng dẫn từ 1 đến 10 mà nó thành mịe cái xe lăn là seo nhỉ )) T bị sai ở khúc nào vậy bây ơii??',
 'Móa sao kì zậy bà',
 'Uh zị zị á bà',
 'Sao hqua có ng kêu ko chua mà',
 'Nivea nghe rõ trả lời nhé brand',
 'Chơi sao chơi méc',
 'M thâm quá m oi',
 'M oi, qá đáng lắm luông',
 'md quá m ơi????',
 'Có thôi đi k',
 'có ai tag bả vô chx:)))',
 'Con này là con lai giữa chó và dê à m.n',
 'Từ khi có AI t đã không tin mọi video',
 'Yea it is but it doesn t live long Probably only days or weeks',
 'mình sẽ học từ 9h sáng tới 11h trưa thì lên ăn cơm bà nấu.',
 'đúng như câu nói "you can take a man out of hanoi, but you can\'t take hanoi out of a man".',
 'T biết bơi mà Cách t bơi ở bể bơi ))))',
 'mi đã bt quá nhiều về kao :)))',
 'sao biet hay z',
 'Ủa t tưởng có mình t làm trò này',
 'Lộ hết bí mật quốc gia r',
 'Các bạn xuống nước ấy thì duỗi chân ,tay ra bơi là đc à dễ mà chứ mình duỗi là hai cái lỗ mũi mình nó uống nước r',
 'Nghiêm trọng, Vercel dịch vụ hosting m

In [95]:
len(thread_raw), len(theanh_clean)

(11219, 601)

In [49]:
save_json(thread_clean, r'C:\Users\ASUS\Desktop\crawl_fb\src\data\clean\thread.json')